# Sklearn Pipeline Best Practices with MLflow

## Topic Goal

This notebook properly implements the expected **Sklearn Pipeline Best Practices** use case with MLflow.

In production machine learning, a model should not be logged alone if it depends on preprocessing.  
The preprocessing logic and the model should be packaged together as one pipeline.

This notebook demonstrates:

1. Why logging only the estimator is risky.
2. How preprocessing outside the model can create production mismatch.
3. How to build a complete `sklearn.pipeline.Pipeline`.
4. How to include `ColumnTransformer`, `StandardScaler`, and `OneHotEncoder`.
5. How to train the complete pipeline.
6. How to log the complete pipeline with MLflow.
7. How to infer model signature from raw input.
8. How to load the logged model using `mlflow.pyfunc.load_model`.
9. How to verify that the loaded model can predict directly on raw input.
10. How to register the complete pipeline model using the same-run `model_uri`.

This is the correct production pattern because the MLflow model contains both preprocessing and prediction logic.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, JSON utilities, and required ML algorithms.

The main focus is on:

```text
ColumnTransformer + Pipeline + MLflow model logging
```


In [ ]:
# Import os to create folders and manage local paths.
import os

# Import json to save pipeline metadata artifacts.
import json

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import datetime to timestamp generated metadata.
from datetime import datetime

# Import MLflow for tracking, model logging, and registry.
import mlflow

# Import MLflow sklearn flavor to log sklearn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input/output schema.
from mlflow.models.signature import infer_signature

# Import pandas for tabular data processing.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import train_test_split for splitting dataset.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for applying different preprocessing to different column groups.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical columns.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical columns.
from sklearn.preprocessing import StandardScaler

# Import Pipeline for combining preprocessing and model into one deployable object.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for churn prediction.
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow and Local Folders

This notebook uses local file-based MLflow tracking.

Generated artifacts will be saved under:

```text
outputs/
```


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "advanced_19_sklearn_pipeline_best_practices"

# Define run name.
RUN_NAME = "19_sklearn_pipeline_best_practices_same_run_usecase"

# Remove inherited MLflow Project experiment ID if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for generated artifacts.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for supporting files.
os.makedirs("artifacts", exist_ok=True)

# Configure MLflow local tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print setup details.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)


## 3. Load Dataset

We use the customer churn dataset.

The model predicts whether a customer is likely to churn.


In [ ]:
# Check whether dataset exists.
if not os.path.exists(DATA_PATH):
    # Raise clear error if dataset is missing.
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Load dataset.
df = pd.read_csv(DATA_PATH)

# Display first five rows.
display(df.head())

# Define target column.
target_column = "churn"

# Define ID column.
id_column = "customer_id"

# Print dataset shape.
print("Dataset shape:", df.shape)


## 4. Prepare Raw Features and Target

The important idea is that `X` contains raw input columns.

This includes categorical columns such as:

```text
gender, region, plan_type, contract_type
```

The final MLflow model should be able to accept this raw input directly.


In [ ]:
# Create raw feature matrix by removing ID and target columns.
X = df.drop(columns=[id_column, target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns from raw input.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns from raw input.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Split raw dataset into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print feature details.
print("Raw feature columns:", X.columns.tolist())
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 5. Bad Practice Demonstration: Preprocessing Outside the Model

This section demonstrates a common production mistake.

Bad practice:

```text
Fit preprocessing separately
Train estimator on transformed data
Log or deploy only the estimator
```

Problem:

```text
The deployed model cannot accept raw customer records directly.
Someone must remember to repeat the exact same preprocessing before prediction.
```

This can lead to training-serving skew.


In [ ]:
# Create a standalone preprocessor.
bad_practice_preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Fit preprocessor only on training data and transform training data.
X_train_transformed = bad_practice_preprocessor.fit_transform(X_train)

# Transform test data separately.
X_test_transformed = bad_practice_preprocessor.transform(X_test)

# Create estimator-only model.
estimator_only_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Train estimator on already-transformed data.
estimator_only_model.fit(X_train_transformed, y_train)

# Predict transformed test data.
bad_predictions = estimator_only_model.predict(X_test_transformed)

# Calculate bad-practice F1-score.
bad_practice_f1 = f1_score(y_test, bad_predictions, zero_division=0)

# Print result.
print("Bad practice estimator-only F1-score:", bad_practice_f1)
print("Important: This estimator cannot directly consume raw input records.")


## 6. Why the Bad Practice Is Risky

This cell intentionally shows the kind of error/mismatch that can happen.

The estimator-only model expects transformed numeric arrays, not raw pandas records.

We catch the error so the notebook continues running.


In [ ]:
# Take raw sample input.
raw_sample_input = X_test.head(2)

# Try to predict directly using estimator-only model.
try:
    # This is expected to fail or produce incorrect behavior because raw input is not preprocessed.
    estimator_only_model.predict(raw_sample_input)

except Exception as error:
    # Print friendly explanation.
    print("Estimator-only prediction on raw input failed as expected.")
    print("Reason:", error)


## 7. Best Practice: Build a Complete Sklearn Pipeline

Best practice:

```text
Pipeline = preprocessing + model
```

This makes the model object production-ready because it can accept raw input records directly.


In [ ]:
# Create production preprocessor.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Create Random Forest model.
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create complete pipeline with preprocessing and model together.
pipeline = Pipeline(steps=[
    # Step 1: preprocess raw input columns.
    ("preprocessor", preprocessor),

    # Step 2: train/predict using model.
    ("model", model),
])

# Train complete pipeline on raw training data.
pipeline.fit(X_train, y_train)

# Predict directly on raw test data.
predictions = pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Create classification report.
report = classification_report(y_test, predictions, zero_division=0)

# Print metrics.
print("Pipeline accuracy:", accuracy)
print("Pipeline precision:", precision)
print("Pipeline recall:", recall)
print("Pipeline F1-score:", f1)
print(report)


## 8. Verify Pipeline Predicts on Raw Input

This is the key proof.

The complete pipeline should successfully predict on raw customer records.


In [ ]:
# Select raw sample records.
raw_sample_input = X_test.head(5)

# Predict directly using complete pipeline.
raw_sample_predictions = pipeline.predict(raw_sample_input)

# Create prediction output dataframe.
sample_prediction_df = raw_sample_input.copy()

# Add predictions.
sample_prediction_df["prediction"] = raw_sample_predictions

# Display sample predictions.
display(sample_prediction_df)

# Print success message.
print("Success: Complete pipeline can predict directly on raw input.")


## 9. Create Pipeline Metadata Artifacts

We create artifacts that explain the pipeline structure.

These artifacts are useful for review and audit.


In [ ]:
# Create pipeline metadata.
pipeline_metadata = {
    "created_at": datetime.now().isoformat(),
    "best_practice": "log_preprocessing_and_model_together",
    "bad_practice": "log_only_estimator_without_preprocessing",
    "pipeline_steps": [step_name for step_name, _ in pipeline.steps],
    "categorical_columns": categorical_columns,
    "numerical_columns": numerical_columns,
    "raw_input_columns": X.columns.tolist(),
    "model_class": model.__class__.__name__,
    "reason": "The logged model should accept raw input records and apply preprocessing internally.",
}

# Save pipeline metadata JSON.
with open("outputs/pipeline_metadata.json", "w") as file:
    json.dump(pipeline_metadata, file, indent=4)

# Save raw sample input.
raw_sample_input.to_csv("outputs/sample_raw_input.csv", index=False)

# Save sample predictions.
sample_prediction_df.to_csv("outputs/sample_raw_predictions.csv", index=False)

# Save classification report.
with open("outputs/classification_report.txt", "w") as file:
    file.write(report)

# Create bad vs best practice comparison table.
practice_comparison = pd.DataFrame([
    {
        "approach": "bad_practice_estimator_only",
        "contains_preprocessing": False,
        "accepts_raw_input_directly": False,
        "risk": "training_serving_skew",
        "f1_score": bad_practice_f1,
    },
    {
        "approach": "best_practice_complete_pipeline",
        "contains_preprocessing": True,
        "accepts_raw_input_directly": True,
        "risk": "reduced",
        "f1_score": f1,
    },
])

# Save practice comparison.
practice_comparison.to_csv("outputs/pipeline_best_practice_comparison.csv", index=False)

# Display comparison.
display(practice_comparison)

# Print artifact names.
print("Saved artifacts:")
print("- outputs/pipeline_metadata.json")
print("- outputs/sample_raw_input.csv")
print("- outputs/sample_raw_predictions.csv")
print("- outputs/classification_report.txt")
print("- outputs/pipeline_best_practice_comparison.csv")


## 10. Infer Model Signature from Raw Input

The signature should be inferred from raw input columns because the pipeline accepts raw input.

This is the correct behavior for production serving.


In [ ]:
# Create input example using raw test data.
input_example = X_test.head(5)

# Generate sample predictions using the full pipeline.
sample_predictions = pipeline.predict(input_example)

# Infer model signature from raw input and predictions.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 11. Same-Run MLflow Logging with Complete Pipeline

This is the production-style logging step.

Inside the **same MLflow run**, we log:

- metrics
- pipeline metadata artifacts
- sample raw input
- sample predictions
- model with preprocessing included
- input example
- signature
- model URI

Then the model is registered using the same-run `model_uri`.


In [ ]:
# Start the SAME MLflow run for pipeline, artifacts, signature, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log pipeline step names.
    mlflow.log_param("pipeline_steps", ",".join([step_name for step_name, _ in pipeline.steps]))

    # Log categorical columns.
    mlflow.log_param("categorical_columns", ",".join(categorical_columns))

    # Log numerical columns.
    mlflow.log_param("numerical_columns", ",".join(numerical_columns))

    # Log best practice flag.
    mlflow.log_param("preprocessing_inside_pipeline", True)

    # Set run purpose tag.
    mlflow.set_tag("run_purpose", "sklearn_pipeline_best_practices")

    # Set best practice tag.
    mlflow.set_tag("best_practice", "log_complete_pipeline_not_estimator_only")

    # Log pipeline metrics.
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Log bad-practice comparison metric.
    mlflow.log_metric("bad_practice_estimator_only_f1", bad_practice_f1)

    # Log artifacts.
    mlflow.log_artifact("outputs/pipeline_metadata.json")
    mlflow.log_artifact("outputs/sample_raw_input.csv")
    mlflow.log_artifact("outputs/sample_raw_predictions.csv")
    mlflow.log_artifact("outputs/classification_report.txt")
    mlflow.log_artifact("outputs/pipeline_best_practice_comparison.csv")

    # Log the complete pipeline with input example and signature.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register model using same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print final details.
print("Same-run model URI:", model_uri)
print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)


## 12. Load Logged Model and Verify Raw Input Prediction

This verifies that the logged MLflow model can predict directly on raw input.

This is the final proof that the pipeline was logged correctly.


In [ ]:
# Load the logged model using MLflow pyfunc.
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Predict using the loaded MLflow model on raw sample input.
loaded_model_predictions = loaded_model.predict(raw_sample_input)

# Create verification dataframe.
verification_df = raw_sample_input.copy()

# Add loaded model predictions.
verification_df["loaded_model_prediction"] = loaded_model_predictions

# Display verification results.
display(verification_df)

# Save verification output.
verification_df.to_csv("outputs/loaded_model_raw_input_verification.csv", index=False)

# Print success message.
print("Success: Loaded MLflow model predicts directly on raw input.")


## 13. Verify Generated Artifacts

This section verifies local artifacts.


In [ ]:
# Define expected artifacts.
expected_artifacts = [
    "outputs/pipeline_metadata.json",
    "outputs/sample_raw_input.csv",
    "outputs/sample_raw_predictions.csv",
    "outputs/classification_report.txt",
    "outputs/pipeline_best_practice_comparison.csv",
    "outputs/loaded_model_raw_input_verification.csv",
]

# Check artifacts.
for artifact_path in expected_artifacts:
    print(artifact_path, "->", os.path.exists(artifact_path))




- In production, the model should not depend on manual preprocessing outside the model package.
- Logging only the estimator can create training-serving skew.
- A complete sklearn pipeline combines preprocessing and model logic.
- `ColumnTransformer` handles numerical and categorical columns in one object.
- `Pipeline` ensures preprocessing happens automatically before prediction.
- MLflow should log the complete pipeline, not only the final estimator.
- The model signature should be inferred from raw input data.
- The loaded MLflow model should successfully predict on raw customer records.
- This is a strong production best practice for sklearn models.
